In [3]:
# import modules
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
#from scipy.stats import vonmises_fisher

# load the result object
from holopy.core.holopy_object import HoloPyObject, FullLoader
from holopy.core.utils import ensure_array, dict_without
import yaml
import importlib
# load using h5py
import h5py as h5

import holopy as hp
from holopy.core.process import normalize, bg_correct, center_find, subimage
from holopy.scattering import Sphere, Spheres, calc_holo
from holopy.inference import prior, ExactModel, CmaStrategy, EmceeStrategy, AlphaModel, NmpfitStrategy
from holopy.inference import model

ImportError: 

IMPORTANT: PLEASE READ THIS FOR ADVICE ON HOW TO SOLVE THIS ISSUE!

Importing the numpy C-extensions failed. This error can happen for
many reasons, often due to issues with your setup or how NumPy was
installed.

We have compiled some common reasons and troubleshooting tips at:

    https://numpy.org/devdocs/user/troubleshooting-importerror.html

Please note and check the following:

  * The Python version is: Python3.9 from "/Users/kaitorrens/miniforge3/envs/holopy-devel/bin/python"
  * The NumPy version is: "1.20.3"

and make sure that they are the versions you expect.
Please carefully study the documentation linked above for further help.

Original error was: dlopen(/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/core/_multiarray_umath.cpython-39-darwin.so, 0x0002): Library not loaded: @rpath/libgfortran.5.dylib
  Referenced from: <D37BED4E-7F75-3467-A281-7E7E316989C9> /Users/kaitorrens/miniforge3/envs/holopy-devel/lib/libopenblas.0.d

In [3]:
#needed to make display work properly (there are other options as well if this fails)
%matplotlib tk

ImportError: 

IMPORTANT: PLEASE READ THIS FOR ADVICE ON HOW TO SOLVE THIS ISSUE!

Importing the numpy C-extensions failed. This error can happen for
many reasons, often due to issues with your setup or how NumPy was
installed.

We have compiled some common reasons and troubleshooting tips at:

    https://numpy.org/devdocs/user/troubleshooting-importerror.html

Please note and check the following:

  * The Python version is: Python3.9 from "/Users/kaitorrens/miniforge3/envs/holopy-devel/bin/python"
  * The NumPy version is: "1.20.3"

and make sure that they are the versions you expect.
Please carefully study the documentation linked above for further help.

Original error was: dlopen(/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/core/_multiarray_umath.cpython-39-darwin.so, 0x0002): Library not loaded: @rpath/libgfortran.5.dylib
  Referenced from: <D37BED4E-7F75-3467-A281-7E7E316989C9> /Users/kaitorrens/miniforge3/envs/holopy-devel/lib/libopenblas.0.d

# Create hologram using known ground truth parameters

In [5]:
# ground truth parameters used to generate a hologram using scattering theory
# optical parameters
medium_index = 1.33
illum_wavelen = 0.660
illum_polarization = (0.56, 0.83)
detector = hp.detector_grid(shape=100, spacing=0.177)

# geometric parameters
N_1_TRUE = 1.59
N_2_TRUE = 1.59
R_1_TRUE = 0.65
R_2_TRUE = 0.65
# originally for spheres on top of each other, changed to less problematic case
X1_TRUE = 5
Y1_TRUE = 5
Z1_TRUE = 5
X2_TRUE = 4
Y2_TRUE = 4
Z2_TRUE = 5

# derived geometric parameters
Xg_TRUE = (X1_TRUE + X2_TRUE)/2
Yg_TRUE = (Y1_TRUE + Y2_TRUE)/2
Zg_TRUE = (Z1_TRUE + Z2_TRUE)/2
GAP = np.sqrt((X1_TRUE-X2_TRUE)**2+(Y1_TRUE-Y2_TRUE)**2+(Z1_TRUE-Z2_TRUE)**2)
# should use absolute value here?
THETA = np.arccos(abs(Z1_TRUE-Z2_TRUE)/GAP)
if (X1_TRUE-X2_TRUE) != 0:
    # should I be using absolute value?
    PHI = np.arctan((Y1_TRUE-Y2_TRUE)/(X1_TRUE-X2_TRUE))
elif (Y1_TRUE-Y2_TRUE) > 0:
    PHI = (np.pi)/2
elif (Y1_TRUE-Y2_TRUE) < 0:
    PHI = -np.pi/2
# both x and y are the same so phi is undefined -> set to 2pi
else:
    PHI = 2*np.pi
# adjust range from -pi to pi into 0 to 2pi
if PHI < 0:
    PHI = 2*np.pi + PHI

In [6]:
print(GAP)
print(THETA)
print(PHI)

1.4142135623730951
1.5707963267948966
0.7853981633974483


In [7]:
# create a hologram from two spheres, with parameters specified above
s1 = Sphere(center=(X1_TRUE, Y1_TRUE, Z1_TRUE), n = N_1_TRUE, r = R_1_TRUE)
s2 = Sphere(center=(X2_TRUE, Y2_TRUE, Z2_TRUE), n = N_2_TRUE, r = R_2_TRUE)

collection = Spheres([s1, s2])
holo1 = calc_holo(detector, collection, medium_index, illum_wavelen,
                 illum_polarization)
# need to specify noise sd if priors not uniform, here use one from Caroline's fit used
# in mcmc fitting notebook
# specifying it in this way seems to lead to errors when loading fits as it appears as
# a coordinate instead of an attribute, but assigning as attrs here also leads to errors
# easiest to assign as coord here and then deal with downstream
# holo1.assign_attrs(noise_sd = 0.00862558)
holo1['noise_sd'] = 0.00862558 
hp.show(holo1)

2025-05-01 14:53:15.855 python[75580:850376] +[IMKClient subclass]: chose IMKClient_Modern
2025-05-01 14:53:15.855 python[75580:850376] +[IMKInputSession subclass]: chose IMKInputSession_Modern


In [ ]:
# create a hologram from two spheres, one above the other
s1 = Sphere(center=(5, 5, 5), n = N_1_TRUE, r = R_1_TRUE)
s2 = Sphere(center=(5, 5, 3.5), n = N_2_TRUE, r = R_2_TRUE)

collection = Spheres([s1, s2])
holo_over = calc_holo(detector, collection, medium_index, illum_wavelen,
                 illum_polarization)
holo_over['noise_sd'] = 0.00862558 
hp.show(holo_over)

In [7]:
# create a hologram from two spheres, one next to the other
s1 = Sphere(center=(5, 5, 5), n = N_1_TRUE, r = R_1_TRUE)
s2 = Sphere(center=(4, 4, 5), n = N_2_TRUE, r = R_2_TRUE)

collection = Spheres([s1, s2])
holo_side = calc_holo(detector, collection, medium_index, illum_wavelen,
                 illum_polarization)
holo_side['noise_sd'] = 0.00862558 
hp.show(holo_side)

# Set parameters used in model fitting/ initialization best guesses

In [8]:
# info for modelling/ fitting

SPACING = 0.177
WAVELEN = 0.660
MEDIUM_INDEX = 1.33
POLARIZATION = POLARIZATION = [0.56, 0.83] # Calibrated 2020-09-02

#Sphere 1
R_1_MEAN = 0.6749016953839639
R_1_SIGMA =  0.00047233977835876834
N_1_MEAN =  1.5848484802283918
N_1_SIGMA =  0.00027320846407879903
#Sphere 2
R_2_MEAN =  0.6446649533295826
R_2_SIGMA =  0.000617682113114549
N_2_MEAN =  1.601784237444771
N_2_SIGMA =  0.0003353994780766957
# not sure what this is and if I should be changing it or not
DIMER_Z_GUESS = 4.20

# Test dummy parameter method where theta and phi are combined in KaiModel object

In [9]:
# define model creation using KaiModel object
def create_kaimodel(parameters):
    s1_r = parameters['r_1']
    s2_r = parameters['r_2']
    s1_n = parameters['n_1']
    s2_n = parameters['n_2']
    center_x = parameters['x_g']
    center_y = parameters['y_g']
    center_z = parameters['z_g']
    theta = parameters['theta']
    phi = parameters['phi']
    gap = parameters['gap']
    alpha = parameters['alpha']
    
    gap_center = np.array([center_x, center_y, center_z])
    components = np.array([np.cos(phi) * np.sin(theta), np.sin(phi) * np.sin(theta), np.cos(theta)])
    s1_center = gap_center + (s1_r + gap/2) * components
    s2_center = gap_center - (s2_r + gap/2) * components
        
    scatterer = Spheres([Sphere(r=s1_r, n=s1_n, center=s1_center),
                         Sphere(r=s2_r, n=s2_n, center=s2_center)], warn=False)
    return model.KaiModel(scatterer, alpha=alpha)

## Skip CMA for now and so hold off on implementing model.generate_guess

In [90]:
dimer_holo = holo1
#x, y = img.center * SPACING
x, y = Xg_TRUE, Yg_TRUE

# Step 1: Fitting the dimer with gap set to 0
# Set priors and run CMAES allowing z, Theta, Phi and alpha to vary

# I may need to modify the bounds on these priors
# recently changed R_1_Mean to R_1_TRUE and same for R2, N1, N2
r_1 = prior.BoundedGaussian(R_1_TRUE, R_1_SIGMA, lower_bound=0, upper_bound=1.0, name="r_1")
r_2 = prior.BoundedGaussian(R_2_TRUE, R_2_SIGMA, lower_bound=0, upper_bound=1.0, name="r_2")
n_1 = prior.BoundedGaussian(N_1_TRUE, N_1_SIGMA, lower_bound=0, upper_bound=1.7, name="n_1")
n_2 = prior.BoundedGaussian(N_2_TRUE, N_2_SIGMA, lower_bound=0, upper_bound=1.7, name="n_2")
x_g = prior.BoundedGaussian(x, SPACING, lower_bound=(x-5), 
                            upper_bound=(x+5), name="x_g")
y_g = prior.BoundedGaussian(y, SPACING, lower_bound=(y-5), 
                            upper_bound=(y+5), name="y_g")
z_g = prior.BoundedGaussian(Zg_TRUE, 1, lower_bound=0, upper_bound=50, name="z_g")
# note: k argument needs to be the same for both theta and phi
theta = prior.Theta(5, THETA, name="theta")
phi = prior.Phi(5, PHI, name="phi")
gap = prior.BoundedGaussian((GAP-R_1_TRUE-R_2_TRUE), 0.005, lower_bound=0, 
                            upper_bound=R_1_TRUE, name="gap")
# changed from 0.8 to 0.997
alpha = prior.BoundedGaussian(0.997, 0.5, lower_bound=0.5, 
                            upper_bound=1.2, name="alpha") 
step_5_parameters = {'r_1': r_1, 'r_2': r_2, 'n_1': n_1, 'n_2': n_2,
                    'x_g': x_g, 'y_g': y_g, 'z_g': z_g,
                    'theta': theta, 'phi': phi, 'gap': gap, 'alpha': alpha}
model5 = create_kaimodel(step_5_parameters)

In [91]:
model5._parameters

[BoundedGaussian(mu=1.59, sd=0.00027320846407879903, lower_bound=0, upper_bound=1.7, name='n_1'),
 BoundedGaussian(mu=0.65, sd=0.00047233977835876834, lower_bound=0, upper_bound=1.0, name='r_1'),
 BoundedGaussian(mu=4.5, sd=0.177, lower_bound=-0.5, upper_bound=9.5, name='x_g'),
 BoundedGaussian(mu=0.1142135623730951, sd=0.005, lower_bound=0, upper_bound=0.65, name='gap'),
 Phi(mu=0.7853981633974483, name='phi', sd=1),
 Theta(mu=1.5707963267948966, name='theta', sd=1),
 BoundedGaussian(mu=4.5, sd=0.177, lower_bound=-0.5, upper_bound=9.5, name='y_g'),
 BoundedGaussian(mu=5.0, sd=1, lower_bound=0, upper_bound=50, name='z_g'),
 BoundedGaussian(mu=1.59, sd=0.0003353994780766957, lower_bound=0, upper_bound=1.7, name='n_2'),
 BoundedGaussian(mu=0.65, sd=0.000617682113114549, lower_bound=0, upper_bound=1.0, name='r_2'),
 BoundedGaussian(mu=0.997, sd=0.5, lower_bound=0.5, upper_bound=1.2, name='alpha')]

## Start by just using the true values as starting points to make sure fits are working as expected

In [92]:
# now generate fit strategy with initial points given by the true values

# originally 50 walkers but start with 30 for speed
nwalkers = 30

# originally used model5.generate_guess(nwalkers, scaling=0.1) but need different method
# before we implement .generate_guess for joint von Mises_Fisher
initial_guess = np.zeros((nwalkers, len(model5._parameters)))
for n in range(nwalkers):
    means = []
    # add some variance in starting point based on variance in distributions
    # need to make sure to avoid unphysical starting positions
    for p in model5._parameters:
        means.append(p.mu)
    # correct phi and theta for continuous angular space ie) 0 to 2pi and 0 to pi
    means[4] = means[4]%(2*np.pi)
    means[5] = means[5]%np.pi
    initial_guess[n,:] = means
emcee_strategy_initial_match_true = EmceeStrategy(npixels=8000, nwalkers=nwalkers, walker_initial_pos=initial_guess)

In [42]:
initial_guess

array([[1.59      , 0.65      , 4.5       , 0.11421356, 0.78539816,
        1.57079633, 4.5       , 5.        , 1.59      , 0.65      ,
        0.8       ],
       [1.59      , 0.65      , 4.5       , 0.11421356, 0.78539816,
        1.57079633, 4.5       , 5.        , 1.59      , 0.65      ,
        0.8       ],
       [1.59      , 0.65      , 4.5       , 0.11421356, 0.78539816,
        1.57079633, 4.5       , 5.        , 1.59      , 0.65      ,
        0.8       ],
       [1.59      , 0.65      , 4.5       , 0.11421356, 0.78539816,
        1.57079633, 4.5       , 5.        , 1.59      , 0.65      ,
        0.8       ],
       [1.59      , 0.65      , 4.5       , 0.11421356, 0.78539816,
        1.57079633, 4.5       , 5.        , 1.59      , 0.65      ,
        0.8       ],
       [1.59      , 0.65      , 4.5       , 0.11421356, 0.78539816,
        1.57079633, 4.5       , 5.        , 1.59      , 0.65      ,
        0.8       ],
       [1.59      , 0.65      , 4.5       , 0.11421356, 0.

In [93]:
# save path for fit with initial conditions given by ground truth values
SAVEPATH = '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/normal_side_by_side_geometry/initial_conditions_match_true_values/von_Mises_Fisher_fit_with_alpha_corrected_1'

## Try to implement reasonable starting points

In [12]:
# now try to actually fit

# originally 50 walkers but start with 30 for speed
nwalkers = 30

# originally used model5.generate_guess(nwalkers, scaling=0.1) but need different method
# before we implement .generate_guess for joint von Mises_Fisher
initial_guess = np.zeros((nwalkers, len(model5._parameters)))
for n in range(nwalkers):
    means = []
    # add some variance in starting point based on variance in distributions
    # need to make sure to avoid unphysical starting positions
    for p in model5._parameters:
        means.append(p.mu + np.random.normal(0,p.sd*0.5))
    # correct phi and theta for continuous angular space ie) 0 to 2pi and 0 to pi
    means[4] = means[4]%(2*np.pi)
    means[5] = means[5]%np.pi
    initial_guess[n,:] = means
emcee_strategy = EmceeStrategy(npixels=8000, nwalkers=nwalkers, walker_initial_pos=initial_guess)

In [13]:
initial_guess

array([[1.59010735, 0.64990483, 4.5536777 , 0.10871318, 0.83998846,
        2.06555865, 4.56764888, 5.56761262, 1.58964961, 0.64972596,
        0.70541683],
       [1.58991183, 0.65013955, 4.54991924, 0.1140885 , 1.25103787,
        1.61350286, 4.4538946 , 5.06189642, 1.5902999 , 0.65022903,
        0.8511737 ],
       [1.58986442, 0.65007   , 4.48166112, 0.11106031, 0.38629928,
        2.40025235, 4.5457841 , 5.59605727, 1.59009078, 0.64982586,
        1.25999053],
       [1.58996874, 0.64951047, 4.60535338, 0.11343974, 0.01941823,
        1.66868829, 4.550893  , 5.46384635, 1.59015377, 0.6504006 ,
        1.03547269],
       [1.58981903, 0.64984986, 4.5338971 , 0.11156589, 0.17595278,
        1.75746655, 4.63191717, 5.69912014, 1.59003683, 0.64985211,
        1.01345943],
       [1.59002337, 0.64987833, 4.70444882, 0.11061617, 1.05599917,
        1.76785193, 4.3815482 , 4.95816406, 1.59012414, 0.64974449,
        0.72002046],
       [1.59004793, 0.6502377 , 4.52623726, 0.11544929, 1.

In [15]:
# save path for random starting conditions
SAVEPATH = '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/normal_side_by_side_geometry/variable_initial_conditions_version_1/von_Mises_Fisher_fit_1'

## Run fit

In [94]:
# make sure to change strategy to desired one
results5 = hp.sample(dimer_holo, model5, strategy=emcee_strategy_initial_match_true)
hp.save(SAVEPATH+'_mcmc.h5', results5)

print('von Mises-Fisher angles fit completed')
print(results5.guess_parameters)
print(results5.parameters)

[dhcp-10-250-113-173.harvard.edu:86790] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-113-173.501/jf.0/2914385920/sm_segment.dhcp-10-250-113-173.501.adb60000.0 could be created.
[dhcp-10-250-113-173.harvard.edu:86796] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-113-173.501/jf.0/3926327296/sm_segment.dhcp-10-250-113-173.501.ea070000.0 could be created.
[dhcp-10-250-113-173.harvard.edu:86795] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-113-173.501/jf.0/3930456064/sm_segment.dhcp-10-250-113-173.501.ea460000.0 could be created.
[dhcp-10-250-113-173.harvard.edu:86791] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-113-173.501/jf.0/2800222208/sm_segment.d

von Mises-Fisher angles fit completed
{'n_1': 1.59, 'r_1': 0.65, 'x_g': 4.5, 'gap': 0.1142135623730951, 'phi': 0.7853981633974483, 'theta': 1.5707963267948966, 'y_g': 4.5, 'z_g': 5.0, 'n_2': 1.59, 'r_2': 0.65, 'alpha': 0.997}
{'n_1': 1.59, 'r_1': 0.65, 'x_g': 4.5, 'gap': 0.1142135623730951, 'phi': 0.7853981633974483, 'theta': 1.5707963267948966, 'y_g': 4.5, 'z_g': 5.0, 'n_2': 1.59, 'r_2': 0.65, 'alpha': 0.997}


In [95]:
# return real fit values
starting_means = []
for p in model5._parameters:
        starting_means.append(p.mu)
print(starting_means)

[1.59, 0.65, 4.5, 0.1142135623730951, 0.7853981633974483, 1.5707963267948966, 4.5, 5.0, 1.59, 0.65, 0.997]


In [18]:
print(dimer_holo.noise_sd)

<xarray.DataArray 'noise_sd' ()>
array(0.00862558)
Coordinates:
    noise_sd  float64 0.008626


In [31]:
# need scipy 1.15 while we have 1.10 is this new version incompatible? -> yes incompatible with parrellel tempering
mu = np.array([-np.sqrt(0.5), -np.sqrt(0.5), 0])
vmf = stats.vonmises_fisher(mu, 5)

AttributeError: module 'scipy.stats' has no attribute 'vonmises_fisher'

In [39]:
-11.425662431912794%(2*np.pi)

1.140708182446378

# Test different geometries

## Test with spheres side by side

## Test with spheres very slightly offset from one above the other

# Visualize fit results

## Load fit if necessary

In [101]:
# path that determines what fit you load
results_path = '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/normal_side_by_side_geometry/variable_initial_conditions_version_1/von_Mises_Fisher_fit_1_mcmc.h5'

In [102]:
# try to load fit result object using hp.load code
# may need to modify code now that noise_sd is assigned as an attribute instead of a coord
attr_coords = '_attr_coords'
def unpack_attrs(a):
    if len(a) == 0:
        return a
    new_attrs={}
    attr_ref = yaml.load(a[attr_coords], Loader=FullLoader)
    attrs_to_ignore = ['spacing', 'name', '_dummy_channel', '_image_scaling']
    for attr in dict_without(attr_ref, attrs_to_ignore):
        if attr_ref[attr]:
            new_attrs[attr] = xr.DataArray(
                a[attr],
                coords=attr_ref[attr],
                dims=list(attr_ref[attr].keys()))
        elif attr in a:
            new_attrs[attr] = yaml.safe_load(a[attr])
        else:
            new_attrs[attr] = None
    return new_attrs

with xr.open_dataset(results_path, engine='h5netcdf') as ds:
    if '_source_class' in ds.attrs:
        _source_class = ds.attrs.pop('_source_class')
        pathtok = _source_class.split('.')
        cls = getattr(importlib.import_module(".".join(pathtok[:-1])), pathtok[-1])
        #ds.close()
        #return_variable = cls._load(results_path)
    #def _load(cls, ds, **kwargs):
        #with xr.open_dataset(ds, engine='h5netcdf', **kwargs) as ds:
        print(ds.load())
        dataset = ds
        data = dataset.data
        data.attrs = unpack_attrs(data.attrs)
        # Kai added this to move noise_sd to correct place
        data.attrs['noise_sd'] = data.coords['noise_sd'].to_numpy()
        if '_flat' in data.attrs.keys():
            flats = np.array(data.attrs['_flat']).T
            levels = [data.original_dims[key] for key in ['x', 'y', 'z']]
            codes = [[level.index(f) for f in flat]
                     for level, flat in zip(levels, flats)]
            flat_index = pd.MultiIndex(levels, codes, names=['x', 'y', 'z'])
            coordnames = list(data.coords)
            coordnames.remove('point')
            # seems like noise_sd should be attribute not coordinate
            coordnames.remove('noise_sd') # added this
            coords = {coord: data[coord] for coord in coordnames}
            coords['flat'] = flat_index
            data = xr.DataArray(data.values, dims=coordnames + ['flat'],
                                coords=coords, attrs=data.attrs)
            print(data)
        print(dataset.attrs['model'])
        # seems like model is a problem because I have an sd attribute in the priors?
        #model = yaml.load(dataset.attrs['model'], Loader=FullLoader)
        strategy = yaml.load(dataset.attrs['strategy'], Loader=FullLoader)
        outlist = [data, model, strategy]
        outlist.append(yaml.safe_load(dataset.attrs['time']))
        kwargs = yaml.safe_load(dataset.attrs['_kwargs'])
        for key in ['lnprobs', 'samples', '_best_fit']:
            try:
                kwargs[key] = getattr(dataset, key)
                kwargs[key].attrs = unpack_attrs(kwargs[key].attrs)
            except AttributeError:
                pass
        outlist.append(kwargs)
        # return args
        args = outlist
        
        #args = cls._unserialize(ds.load())
        return_variable = cls(*args)
'''
def _unserialize(cls, dataset):
        data = dataset.data
        data.attrs = unpack_attrs(data.attrs)
        if '_flat' in data.attrs.keys():
            flats = np.array(data.attrs['_flat']).T
            levels = [data.original_dims[key] for key in ['x', 'y', 'z']]
            codes = [[level.index(f) for f in flat]
                     for level, flat in zip(levels, flats)]
            flat_index = pd.MultiIndex(levels, codes, names=['x', 'y', 'z'])
            coordnames = list(data.coords)
            coordnames.remove('point')
            coords = {coord: data[coord] for coord in coordnames}
            coords['flat'] = flat_index
            data = xr.DataArray(data.values, dims=coordnames + ['flat'],
                                coords=coords, attrs=data.attrs)
        model = yaml.load(dataset.attrs['model'], Loader=FullLoader)
        strategy = yaml.load(dataset.attrs['strategy'], Loader=FullLoader)
        outlist = [data, model, strategy]
        outlist.append(yaml.safe_load(dataset.attrs['time']))
        kwargs = yaml.safe_load(dataset.attrs['_kwargs'])

        for key in ['lnprobs', 'samples', '_best_fit']:
            try:
                kwargs[key] = getattr(dataset, key)
                kwargs[key].attrs = unpack_attrs(kwargs[key].attrs)
            except AttributeError:
                pass
        outlist.append(kwargs)
        return outlist
'''

<xarray.Dataset>
Dimensions:    (point: 8000, walker: 30, chain: 1000, parameter: 11)
Coordinates:
    noise_sd   float64 0.008626
  * point      (point) int64 0 1 2 3 4 5 6 ... 7994 7995 7996 7997 7998 7999
  * parameter  (parameter) object 'n_1' 'r_1' 'x_g' ... 'n_2' 'r_2' 'alpha'
Dimensions without coordinates: walker, chain
Data variables:
    data       (point) float64 0.9862 0.9987 1.065 1.578 ... 1.006 1.006 1.218
    lnprobs    (walker, chain) float64 -4.711e+05 -4.711e+05 ... 3.032e+04
    samples    (walker, chain, parameter) float64 1.59 0.6499 ... 0.6503 0.9968
Attributes:
    model:     !KaiModel\n_dummy_scatterer: !Spheres\n  scatterers: [!Sphere ...
    strategy:  !EmceeStrategy\nnwalkers: 30\nnsamples: 1000\nnpixels: 8000\nw...
    time:      1235.4122138023376
    _kwargs:   {}\n
<xarray.DataArray (flat: 8000)>
array([0.98624148, 0.99868092, 1.06543438, ..., 1.00621924, 1.00585964,
       1.21791229])
Coordinates:
  * flat     (flat) object MultiIndex
  * x        (fla

"\ndef _unserialize(cls, dataset):\n        data = dataset.data\n        data.attrs = unpack_attrs(data.attrs)\n        if '_flat' in data.attrs.keys():\n            flats = np.array(data.attrs['_flat']).T\n            levels = [data.original_dims[key] for key in ['x', 'y', 'z']]\n            codes = [[level.index(f) for f in flat]\n                     for level, flat in zip(levels, flats)]\n            flat_index = pd.MultiIndex(levels, codes, names=['x', 'y', 'z'])\n            coordnames = list(data.coords)\n            coordnames.remove('point')\n            coords = {coord: data[coord] for coord in coordnames}\n            coords['flat'] = flat_index\n            data = xr.DataArray(data.values, dims=coordnames + ['flat'],\n                                coords=coords, attrs=data.attrs)\n        model = yaml.load(dataset.attrs['model'], Loader=FullLoader)\n        strategy = yaml.load(dataset.attrs['strategy'], Loader=FullLoader)\n        outlist = [data, model, strategy]\n     

In [84]:
# seems like don't need model at least for current analysis
# end up with noise_sd as a coordinate which is weird
print(return_variable)
samples = return_variable.samples[:,999]
lnprob = return_variable.lnprobs[5]
# can get rid of noise_sd coord using .reset_coords('noise_sd', drop = True)
print(samples.reset_coords('noise_sd',drop=True))
print(lnprob.reset_coords('noise_sd',drop=True))
burnt_samples = return_variable.burn_in(110).samples[:,889]

SamplingResult(data=<xarray.DataArray (flat: 8000)>
array([0.9711212 , 0.94684798, 0.97734354, ..., 0.99462822, 0.97067491,
       0.9919057 ])
Coordinates:
  * flat     (flat) object MultiIndex
  * x        (flat) float64 15.75 3.717 5.31 13.1 ... 7.788 2.124 13.98 16.28
  * y        (flat) float64 6.903 6.903 16.46 2.655 ... 14.16 13.1 7.434 7.434
  * z        (flat) int64 0 0 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0 0 0
Attributes:
    _flat:               [[15.752999999999998, 6.903, 0], [3.7169999999999996...
    illum_polarization:  <xarray.DataArray (vector: 3)>\narray([0.55930131, 0...
    illum_wavelen:       0.66
    medium_index:        1.33
    noise_sd:            0.00862558
    original_dims:       {'x': [0.0, 0.177, 0.354, 0.5309999999999999, 0.708,..., model=<module 'holopy.inference.model' from '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holopy_code/holopy/holopy/inference/model.py'>, strategy=EmceeStrategy(nwalkers=30, nsamples=1000, npixels=8000, w

In [103]:
# set results equal to loaded fit for further analysis
results5 = return_variable

## Work on futher data processing (ie drop pre-burn in data, drop non-converged fits, and decimate remaining data so its independent)

In [104]:
samples = results5.samples
print(samples[:,999][0])
print(means)

<xarray.DataArray 'samples' (parameter: 11)>
array([1.59205666, 0.65042831, 4.50054367, 0.10786519, 0.7859097 ,
       1.56807037, 4.5008076 , 5.00270147, 1.59078549, 0.65046919,
       0.99723374])
Coordinates:
    noise_sd   float64 0.008626
  * parameter  (parameter) object 'n_1' 'r_1' 'x_g' ... 'n_2' 'r_2' 'alpha'
Attributes:
    acceptance_fraction:  0.3671999999999999
[1.59, 0.65, 4.5, 0.1142135623730951, 0.7853981633974483, 1.5707963267948966, 4.5, 5.0, 1.59, 0.65, 0.997]


In [19]:
print(initial_guess[0])

[1.59010735 0.64990483 4.5536777  0.10871318 0.83998846 2.06555865
 4.56764888 5.56761262 1.58964961 0.64972596 0.70541683]


In [105]:
# select gap parameter values for first walker (all chains)
gaps_example = samples[1].sel(parameter='gap')
print(gaps_example)

<xarray.DataArray 'samples' (chain: 1000)>
array([0.1140885 , 0.1140885 , 0.1140885 , 0.1140885 , 0.1140885 ,
       0.1140885 , 0.1140885 , 0.1140885 , 0.11465715, 0.11465715,
       0.11465715, 0.11465715, 0.11465715, 0.11465715, 0.11465715,
       0.11465715, 0.11456956, 0.11422793, 0.11430698, 0.11430698,
       0.11430698, 0.11430698, 0.11430698, 0.11424776, 0.11424776,
       0.11432974, 0.11432974, 0.11432974, 0.11432974, 0.11432974,
       0.1143477 , 0.1143477 , 0.1143477 , 0.1143477 , 0.1143477 ,
       0.1143477 , 0.1143477 , 0.11434372, 0.11432017, 0.11432017,
       0.11432017, 0.11432017, 0.1143191 , 0.11429546, 0.11428694,
       0.11409419, 0.11409419, 0.11413383, 0.11413383, 0.11413383,
       0.11412764, 0.11412421, 0.11412421, 0.11419978, 0.11419978,
       0.11419978, 0.11419978, 0.11417549, 0.11417549, 0.11416665,
       0.11416665, 0.11416665, 0.11416665, 0.11416373, 0.11416373,
       0.11416373, 0.11404824, 0.11404824, 0.11394301, 0.11394301,
       0.11385696, 

In [106]:
plt.plot(gaps_example)

### Drop pre-burn in (right now ad hoc but come back to make more systematic)

In [96]:
# plot pre-burn in
for i in range(4):
    plt.plot(results5.lnprobs[i])

In [107]:
# Use .burn_in() to chop off data before a specific sample number
burnt_results5 = results5.burn_in(140) #120 seems good for this specific fit
for i in range(4):
    plt.plot(burnt_results5.lnprobs[i])
# to do more systematically could maybe calculate some sort of slope vs maximum slope cutoff?

In [38]:
# looking at the fits for all the different walkers it's clear that several don't converge well
for i in range(11):
    plt.plot(burnt_results5.lnprobs[i])

In [108]:
new_sample_length = len(burnt_results5.lnprobs[i])
print(burnt_results5.lnprobs[i][new_sample_length-1])

<xarray.DataArray 'lnprobs' ()>
array(30334.09068016)
Coordinates:
    noise_sd  float64 0.008626
Attributes:
    acceptance_fraction:  0.3671999999999999


### Visualize Data Traces

In [40]:
# look at some traces of theta
for i in range(4):
    plt.plot(samples[i].sel(parameter='theta'))

In [42]:
# look at some traces of phi
for i in range(4):
    plt.plot(samples[i].sel(parameter='phi'))

In [109]:
# look at some traces of gap
for i in range(4):
    plt.plot(samples[i].sel(parameter='gap'))

In [112]:
# look at some traces of r1
for i in range(4):
    plt.plot(samples[i].sel(parameter='r_1'))

In [114]:
# look at some traces of n1
for i in range(4):
    plt.plot(samples[i].sel(parameter='n_1'))

In [51]:
# look at some of the traces for walkers that don't converge
plt.plot(samples[7].sel(parameter='theta'))
plt.plot(samples[10].sel(parameter='theta'))

In [46]:
# look at theta that did converge and add pi to them
for i in range(4):
    plt.plot(samples[i].sel(parameter='theta')+np.pi)

In [54]:
plt.plot(samples[7].sel(parameter='phi'))
plt.plot(samples[10].sel(parameter='phi'))

In [57]:
plt.plot(samples[7].sel(parameter='gap'))
plt.plot(samples[10].sel(parameter='gap'))

### Drop bad convergence (ie low lnprob) walkers. Later attempt to fix these fits instead.

In [46]:
# The bad fits are the swapped angles fits (visible in the reorganized pandas dataset)
# -> is there a good way to correct these or should I just drop them?
# Start by implementing a cutoff in lnprobs to drop them and then can work on fixing later
samples = burnt_results5.samples
converged_samples_nan = xr.DataArray()
for i in range(len(samples)):
    new_sample_length = len(burnt_results5.lnprobs[i])
    if burnt_results5.lnprobs[i][new_sample_length-1] > 0:
        converged_samples_nan = xr.concat([converged_samples_nan,samples[i]],'walker')
        # .append() isn't quite what we want, try to use xarray methods
    converged_samples = converged_samples_nan[1:]
print(converged_samples)

<xarray.DataArray (walker: 28, chain: 860, parameter: 11)>
array([[[1.59012336, 0.64992244, 4.50818631, ..., 1.59013892,
         0.65004984, 0.93766197],
        [1.59012336, 0.64992244, 4.50818631, ..., 1.59013892,
         0.65004984, 0.93766197],
        [1.59012336, 0.64992244, 4.50818631, ..., 1.59013892,
         0.65004984, 0.93766197],
        ...,
        [1.59205666, 0.65042831, 4.50054367, ..., 1.59078549,
         0.65046919, 0.99723374],
        [1.59205666, 0.65042831, 4.50054367, ..., 1.59078549,
         0.65046919, 0.99723374],
        [1.59205666, 0.65042831, 4.50054367, ..., 1.59078549,
         0.65046919, 0.99723374]],

       [[1.5900825 , 0.64990177, 4.51373969, ..., 1.59012994,
         0.65007001, 0.95339774],
        [1.59008451, 0.64989982, 4.51377521, ..., 1.59013043,
         0.65007067, 0.95285207],
        [1.59008451, 0.64989982, 4.51377521, ..., 1.59013043,
         0.65007067, 0.95285207],
...
        [1.59218846, 0.65023294, 4.50110299, ..., 1.590840

In [ ]:
# could also do this with ds.drop_sel(space=["IN", "IL"]) where we use walker = [drop indexes]

In [111]:
# plot converged samples
# need to modify this for each fit you want to use it for
# converged_id = [0,1,2,3,4,5,6,8,9]
for id in converged_id:
    plt.plot(burnt_results5.lnprobs[id])

### Decimate data so just keep independent fits

In [47]:
# look at autocorrelation to see how long it is before lose memory so can decimate into independent samples
series = pd.Series(converged_samples[1].sel(parameter='gap'))
autocorr_as_function_of_time = []
for i in range(len(series)):
    autocorr = series.autocorr(lag=i)
    autocorr_as_function_of_time.append(autocorr)
plt.plot(autocorr_as_function_of_time)

/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2634: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2493: RuntimeWarning: divide by zero encountered in true_divide
  c *= np.true_divide(1, fact)


In [16]:
# look at what corresponding trace looks like
plt.plot(converged_samples[1].sel(parameter='gap'))

In [48]:
# look at autocorrelation of gap data from different walkers
for n in range(len(converged_samples)):
    series = pd.Series(converged_samples[n].sel(parameter='gap'))
    autocorr_as_function_of_time = []
    for i in range(len(series)):
        autocorr = series.autocorr(lag=i)
        autocorr_as_function_of_time.append(autocorr)
    plt.plot(autocorr_as_function_of_time)

In [31]:
# look at trace of gap from different walkers
for n in range(len(converged_samples)):
    plt.plot(converged_samples[n].sel(parameter='gap'))

In [35]:
# look at autocorrelation of theta data from different walkers
for n in range(len(converged_samples)):
    series = pd.Series(converged_samples[n].sel(parameter='theta'))
    autocorr_as_function_of_time = []
    for i in range(len(series)):
        autocorr = series.autocorr(lag=i)
        autocorr_as_function_of_time.append(autocorr)
    plt.plot(autocorr_as_function_of_time)

In [36]:
# look at trace of theta from different walkers
for n in range(len(converged_samples)):
    plt.plot(converged_samples[n].sel(parameter='theta'))

In [129]:
# We can use the following notation to cycle throught the different parameter labels
for parameter in converged_samples.coords['parameter'].data:
    print(parameter)

n_1
r_1
x_g
gap
phi
theta
y_g
z_g
n_2
r_2
alpha


In [49]:
# Plot autocorrelation for all the different parameters
# set up plotting
number_columns = int(np.ceil(len(converged_samples.coords['parameter'].data)/3))
fig,axes = plt.subplots(3,number_columns)
m = 1
# go through analysis
for parameter_name in converged_samples.coords['parameter'].data:
    for n in range(len(converged_samples)):
        series = pd.Series(converged_samples[n].sel(parameter=parameter_name))
        autocorr_as_function_of_time = []
        for i in range(len(series)):
            autocorr = series.autocorr(lag=i)
            autocorr_as_function_of_time.append(autocorr)
        plt.subplot(3,number_columns,m)
        plt.plot(autocorr_as_function_of_time)
    # label plot
    row = int(np.floor((m-1)/4))
    column = int(m-4*row)
    axes[row,column-1].set_title(parameter_name)
    fig.supxlabel('Chain Number')
    fig.supylabel('Autocorrelation')
    # increment number tracker
    m = m + 1

In [50]:
# find the correlation time for each of these parameters (ie when autocorrelation drops to 0 for the first time)
all_parameter_indices = []
for parameter_name in converged_samples.coords['parameter'].data:
    all_walker_indices = []
    for n in range(len(converged_samples)):
        series = pd.Series(converged_samples[n].sel(parameter=parameter_name))
        autocorr_as_function_of_time = []
        for i in range(len(series)):
            autocorr = series.autocorr(lag=i)
            autocorr_as_function_of_time.append(autocorr)
            if autocorr < 0:
                index = i
                all_walker_indices.append(index)
                break
    all_parameter_indices.append(all_walker_indices)
print(all_parameter_indices)
# Q: should I find max correlation time or mean correlation time for each parameter?
# start with easiest which is just overall max
overall_correlation_time = np.max(all_parameter_indices)
print(overall_correlation_time)

[[560, 566, 562, 571, 576, 559, 696, 567, 549, 535, 573, 566, 618, 547, 692, 562, 558, 687, 575, 587, 578, 583, 566, 532, 575, 553, 562, 553], [202, 208, 204, 191, 191, 207, 195, 198, 195, 203, 195, 197, 200, 203, 198, 194, 199, 198, 193, 192, 196, 201, 197, 204, 201, 204, 203, 196], [299, 352, 330, 343, 319, 329, 345, 328, 300, 333, 353, 310, 355, 378, 339, 275, 369, 324, 350, 271, 334, 323, 337, 336, 334, 383, 317, 329], [239, 245, 239, 228, 225, 241, 225, 236, 231, 242, 231, 232, 238, 244, 238, 229, 237, 229, 230, 226, 230, 236, 231, 239, 235, 242, 239, 233], [146, 198, 172, 201, 157, 175, 135, 147, 150, 157, 148, 167, 119, 133, 152, 146, 151, 148, 188, 144, 182, 151, 157, 169, 151, 141, 144, 143], [287, 306, 282, 282, 280, 291, 262, 276, 257, 284, 268, 276, 261, 264, 301, 261, 252, 289, 260, 249, 283, 294, 267, 268, 265, 294, 277, 268], [334, 341, 333, 326, 294, 423, 322, 343, 311, 352, 343, 337, 391, 384, 349, 320, 330, 298, 329, 328, 321, 338, 404, 326, 341, 342, 329, 343], [636,

In [51]:
# other approach where we find the mean and then take the max
mean_parameter_corr_time = np.mean(all_parameter_indices, axis=1)
max_of_mean_parameter_corr_time = int(np.max(mean_parameter_corr_time))
print(max_of_mean_parameter_corr_time)

741


In [52]:
# now use correlation time to decimate the data
chain_number = len(converged_samples[0])
number_ind_samples_per_walker = int(np.ceil(chain_number/overall_correlation_time))
independent_samples =[]
for i in range(number_ind_samples_per_walker):
    index = (chain_number-1-overall_correlation_time*i)
    if i==0:
        independent_samples = converged_samples[:,index]
    else:
        independent_samples = xr.concat([independent_samples,(converged_samples[:,index])], 'walker')

In [53]:
# now convert independent samples into a form that can be input into seaborn pairplot
independent_samples_pd = independent_samples.to_dataframe(name = 'independent samples')
independent_samples_pd = independent_samples_pd.reset_index()
independent_samples_pd = independent_samples_pd.pivot('walker','parameter','independent samples')
#print(independent_samples_pd)

/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_75580/3190556392.py:4: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  independent_samples_pd = independent_samples_pd.pivot('walker','parameter','independent samples')


In [54]:
sns.pairplot(independent_samples_pd)

In [56]:
# seems like the parameter sets from the end of the run and the beginning of the run
# cluster in different ways, lets try seperating them

# get parameters at the end of the fitting process
end_of_run_ind = independent_samples[:len(converged_samples)]
end_of_run_ind_pd = end_of_run_ind.to_dataframe(name = 'independent samples')
end_of_run_ind_pd = end_of_run_ind_pd.reset_index()
end_of_run_ind_pd = end_of_run_ind_pd.pivot('walker','parameter','independent samples')
print(end_of_run_ind_pd)

# get parameters from earlier on in the fitting process (~correlation time before the end)
early_run_ind = independent_samples[len(converged_samples):]
early_run_ind_pd = early_run_ind.to_dataframe(name = 'independent samples')
early_run_ind_pd = early_run_ind_pd.reset_index()
early_run_ind_pd = early_run_ind_pd.pivot('walker','parameter','independent samples')
#print(early_run_ind_pd)

parameter     alpha       gap       n_1       n_2       phi       r_1  \
walker                                                                  
0          0.997234  0.107865  1.592057  1.590785  0.785910  0.650428   
1          0.997304  0.108050  1.591769  1.590708  0.786166  0.650521   
2          0.996536  0.107870  1.592021  1.590775  0.785474  0.650441   
3          0.996993  0.107722  1.591921  1.590746  0.785774  0.650509   
4          0.997160  0.108245  1.592090  1.590804  0.785759  0.650363   
5          0.996510  0.109483  1.592285  1.590883  0.786245  0.650119   
6          0.997008  0.108220  1.591938  1.590757  0.785904  0.650421   
7          0.996743  0.108149  1.592037  1.590784  0.785712  0.650395   
8          0.997540  0.107520  1.591684  1.590671  0.786072  0.650619   
9          0.997262  0.108991  1.591573  1.590673  0.785975  0.650480   
10         0.996848  0.108579  1.592058  1.590798  0.785534  0.650323   
11         0.996619  0.109140  1.592267  1.590869  

/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_75580/2063201981.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  end_of_run_ind_pd = end_of_run_ind_pd.pivot('walker','parameter','independent samples')
/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_75580/2063201981.py:15: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  early_run_ind_pd = early_run_ind_pd.pivot('walker','parameter','independent samples')


In [57]:
sns.pairplot(end_of_run_ind_pd)
# hmm seems like one of the fits is comparatively bad and is an outlier

In [27]:
sns.pairplot(early_run_ind_pd)

In [74]:
# compare average of end points of converged fits with ground truth values
mean_prediction = np.mean(end_of_run_ind_pd,axis=0)
print(mean_prediction)
print(starting_means)
mean_differences = np.zeros(len(starting_means))
corresponding_id = [2,5,8,1,4,7,9,10,3,6,0]
for i in range(len(starting_means)):
    mean_differences[i] = starting_means[i]-mean_prediction[corresponding_id[i]]
print(mean_differences)

parameter
alpha    0.996944
gap      0.108336
n_1      1.591999
n_2      1.590778
phi      0.785823
r_1      0.650386
r_2      0.650533
theta    1.567133
x_g      4.500696
y_g      4.501285
z_g      5.003566
dtype: float64
[1.59, 0.65, 4.5, 0.1142135623730951, 0.7853981633974483, 1.5707963267948966, 4.5, 5.0, 1.59, 0.65, 0.8]
[-0.00199858 -0.00038636 -0.00069632  0.00587724 -0.00042445  0.00366343
 -0.00128452 -0.00356586 -0.00077801 -0.00053318 -0.19694439]


In [79]:
mean_prediction = np.mean(end_of_run_ind_pd,axis=0)
mean_differences = mean_prediction.copy()
opp_corresponding_id = [10,3,0,8,4,1,9,5,2,6,7]
for i in range(len(starting_means)):
    mean_differences[i] = starting_means[opp_corresponding_id[i]]-mean_prediction[i]
print(mean_prediction)
print(mean_differences)

parameter
alpha    0.996944
gap      0.108336
n_1      1.591999
n_2      1.590778
phi      0.785823
r_1      0.650386
r_2      0.650533
theta    1.567133
x_g      4.500696
y_g      4.501285
z_g      5.003566
dtype: float64
parameter
alpha   -0.196944
gap      0.005877
n_1     -0.001999
n_2     -0.000778
phi     -0.000424
r_1     -0.000386
r_2     -0.000533
theta    0.003663
x_g     -0.000696
y_g     -0.001285
z_g     -0.003566
dtype: float64
